# Genome-wide QC + thinning + merge (interactive, batched)

All 22 autosomes, same QC + thinning `chr22_qc_thinning_timing.ipynb` validated
(min MAF 1%, HWE 1e-6, missingness < 5%, biallelic-only, `--thin 0.2` fixed --
chr22's calibration), run interactively in this notebook rather than as
submitted batch jobs -- that's the deliberate next step after this, once this
run's timing/behavior is validated. Chromosomes run concurrently (capped, biggest
first) rather than one at a time: chr22 alone took ~10 min for QC, and a fully
serial 22-chromosome run extrapolates to ~9.4 hours -- well over budget.
Concurrency within this single interactive session is the first lever; job
submission (`dsub`, one Google Batch task per chromosome, no shared machine or
network mount) is the next one, deferred until this run's numbers are in.

Once all 22 chromosomes are QC'd/thinned, the last section merges them into a
single genome-wide pfile (and a PLINK1 bed/bim/fam export) -- the actual input
`03_grm_shards`'s GRM construction step needs, since relatedness is computed
against the whole genome at once, not per chromosome.</cell id="061cb961">


## Compute resource

Running `N_CONCURRENT` chromosomes at once, each with `THREADS_PER_JOB` plink2
threads, needs `N_CONCURRENT x THREADS_PER_JOB` vCPUs actually available --
undersizing the environment just serializes the "concurrent" jobs against each
other. Defaults below (4 concurrent x 4 threads = 16 vCPUs) target a 16-32 vCPU /
64-128 GB environment; size the cloud environment to that **before** running the
driver cell, then adjust `N_CONCURRENT`/`THREADS_PER_JOB` to match whatever was
actually provisioned. All 22 chromosomes read from the same network-mounted ACAF
source concurrently here (unlike separate `dsub` Batch workers, which each get
their own network path) -- if wall-clock comes in much worse than
`chr1_time / N_CONCURRENT`-ish, that's the network mount contending, not CPU.

## Setup

Same plink2 install as `chr22_qc_thinning_timing.ipynb`.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version

nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Paths

Same derivation as `chr22_qc_thinning_timing.ipynb` -- see that notebook for the
`--set-all-var-ids`-before-`--rm-dup` rationale (ACAF's pvar ships with unset
variant IDs) and the native-pgen-over-`plink_bed` lesson.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
os.makedirs(BUCKET_DIR, exist_ok=True)

ROUND2B_KEEP_PATH = f"{ANCESTRY_BUCKET_DIR}/reverse_pca_aou/round2b_keep_ids_aou_fit_r2b_p99.9999.txt"
assert os.path.isfile(ROUND2B_KEEP_PATH), f"round 2b keep-list not found: {ROUND2B_KEEP_PATH!r}"

AOU_ROOT = os.path.dirname(WORKSPACE_BUCKET)
ACAF_PGEN_DIR = f"{AOU_ROOT}/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
assert os.path.isdir(ACAF_PGEN_DIR), f"ACAF_PGEN_DIR does not exist: {ACAF_PGEN_DIR!r}"
ACAF_PGEN_PATTERN = ACAF_PGEN_DIR + "/acaf_threshold.chr%d"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

for chr_num in range(1, 23):
    chr_pfile = ACAF_PGEN_PATTERN % chr_num
    for ext in ("pgen", "pvar", "psam"):
        assert os.path.isfile(f"{chr_pfile}.{ext}"), f"missing ACAF input: {chr_pfile}.{ext}"

print("All 22 autosome pgen/pvar/psam trios found.")

## Run parameters

`THIN_P = 0.2`: fixed from `chr22_qc_thinning_timing.ipynb`'s calibration --
not recomputed per chromosome here.

`CHR_LENGTHS`: GRCh38 chromosome lengths (bp), used only to order jobs
biggest-first -- the point isn't precision, it's avoiding a batch that pairs
two large chromosomes together while a batch of small ones finishes early and
sits idle.

In [ ]:
THIN_P = 0.2
N_CONCURRENT = 4      # match to the cloud environment's actual vCPU count -- see "Compute resource"
THREADS_PER_JOB = 4

CHR_LENGTHS = {
    1: 248_956_422, 2: 242_193_529, 3: 198_295_559, 4: 190_214_555,
    5: 181_538_259, 6: 170_805_979, 7: 159_345_973, 8: 145_138_636,
    9: 138_394_717, 10: 133_797_422, 11: 135_086_622, 12: 133_275_309,
    13: 114_364_328, 14: 107_043_718, 15: 101_991_189, 16: 90_338_345,
    17: 83_257_441, 18: 80_373_285, 19: 58_617_616, 20: 64_444_167,
    21: 46_709_983, 22: 50_818_468,
}
CHRS_LARGEST_FIRST = sorted(CHR_LENGTHS, key=CHR_LENGTHS.get, reverse=True)
print(CHRS_LARGEST_FIRST)

## Driver

One QC + thin + persist-to-bucket script per chromosome, run via a
`ThreadPoolExecutor` capped at `N_CONCURRENT` -- each `subprocess.run` blocks
its worker thread on the underlying `plink2` process, so this is really
OS-level process concurrency, not Python threading. Each chromosome's plink2
stdout/stderr goes to its own log file in `LOCAL_WORK_DIR`, not the notebook's
own output, since interleaved concurrent output across 4 processes would be
unreadable.

QC and `--thin` run as a single plink2 call rather than two (unlike
`chr22_qc_thinning_timing.ipynb`'s separate Step 2/Step 3) -- plink2 applies
`--thin` after the other site-level filters regardless of argument order, so
the surviving variant set is identical either way. The only thing this changes
is that the QC'd-but-unthinned intermediate pgen is never written or copied to
the bucket at all -- across 22 concurrently-running chromosomes that
intermediate is the larger of the two outputs, and skipping it cuts both local
I/O and the bucket copy, which is the point of merging the calls here.
`chr22_qc_thinning_timing.ipynb` deliberately keeps the two steps separate
instead, since retuning `THIN_P` there means rerunning only the cheap Step 3,
not the expensive Step 2 -- that tradeoff doesn't apply here now that `THIN_P`
is already fixed from that calibration.

**Resume safety:** `qc_thin_persist` checks `BUCKET_DIR` (not `LOCAL_WORK_DIR`,
which is local scratch and not guaranteed to survive a restart) for an
already-persisted `.pgen`/`.pvar`/`.psam` trio before running plink2 -- if
found, it skips straight to re-deriving `n_thin` from the persisted `.pvar` and
returns `skipped: True`. Re-running this cell after killing a stuck or
oversubscribed run (e.g. to fix `N_CONCURRENT`/`THREADS_PER_JOB`) only
reprocesses chromosomes that hadn't finished yet.

In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def qc_thin_persist(chr_num):
    chr_pfile = ACAF_PGEN_PATTERN % chr_num
    thin_prefix = os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_round2b_thinned")
    thin_name = os.path.basename(thin_prefix)
    bucket_prefix = os.path.join(BUCKET_DIR, thin_name)
    log_path = os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}.log")

    # resume safety: BUCKET_DIR (unlike LOCAL_WORK_DIR) survives a restart, so a
    # chromosome whose trio already landed there is done -- skip re-running plink2
    # on it and just re-derive its count from the persisted .pvar
    if all(os.path.isfile(f"{bucket_prefix}.{ext}") for ext in ("pgen", "pvar", "psam")):
        n_thin = int(subprocess.run(
            ["bash", "-c", f"grep -vc '^##' '{bucket_prefix}.pvar'"],
            capture_output=True, text=True, check=True
        ).stdout.strip())
        return {
            "chr": chr_num, "elapsed_sec": 0.0, "returncode": 0,
            "n_thin": n_thin, "log_path": log_path, "skipped": True,
        }

    script = f'''
set -e
plink2 \
  --pfile "{chr_pfile}" \
  --keep "{ROUND2B_KEEP_PATH}" \
  --thin {THIN_P} \
  --set-all-var-ids '@:#:$r:$a' \
  --new-id-max-allele-len 1000 \
  --maf 0.01 \
  --hwe 1e-6 0.001 keep-fewhet \
  --geno 0.05 \
  --max-alleles 2 \
  --rm-dup exclude-all \
  --nonfounders \
  --threads {THREADS_PER_JOB} \
  --make-pgen \
  --out "{thin_prefix}"

mkdir -p "{BUCKET_DIR}"
cp "{thin_prefix}".pgen "{thin_prefix}".pvar "{thin_prefix}".psam "{thin_prefix}".log "{BUCKET_DIR}/"

grep -vc '^##' "{thin_prefix}.pvar"
'''

    start = time.monotonic()
    with open(log_path, "w") as log_file:
        result = subprocess.run(
            ["bash", "-c", script],
            stdout=log_file, stderr=subprocess.STDOUT
        )
    elapsed = time.monotonic() - start

    n_thin = None
    if result.returncode == 0:
        with open(log_path) as f:
            lines = [l.strip() for l in f if l.strip().isdigit()]
        if len(lines) >= 1:
            n_thin = int(lines[-1])

    return {
        "chr": chr_num, "elapsed_sec": elapsed, "returncode": result.returncode,
        "n_thin": n_thin, "log_path": log_path, "skipped": False,
    }

overall_start = time.monotonic()
results = []
with ThreadPoolExecutor(max_workers=N_CONCURRENT) as pool:
    futures = {pool.submit(qc_thin_persist, chr_num): chr_num for chr_num in CHRS_LARGEST_FIRST}
    for future in as_completed(futures):
        r = future.result()
        if r["skipped"]:
            status = "skipped (already in bucket)"
        else:
            status = "ok" if r["returncode"] == 0 else f"FAILED (see {r['log_path']})"
        print(f"chr{r['chr']:>2}: {r['elapsed_sec']/60:5.1f} min, "
              f"thinned={r['n_thin']}, {status}")
        results.append(r)

overall_elapsed = time.monotonic() - overall_start
print(f"\nTotal wall-clock: {overall_elapsed/3600:.2f} hours")

## Summary

Any chromosome with `returncode != 0` failed -- check its `log_path` before
trusting the run; a partial genome-wide result with a silently-missing
chromosome is worse than an obvious failure. Sum `n_thin` across all 22 rows
for the actual genome-wide variant count (vs. the ~1M target `chr22_qc_thinning_timing.ipynb`
calibrated `THIN_P` toward) -- if it's off by a lot, that's the "chr22 density
isn't necessarily representative" caveat from that notebook showing up for
real, and `THIN_P` may need adjusting and re-running.

Compare `overall_elapsed` against the naive serial estimate
(`sum(elapsed_sec) / 3600`) to see how much the concurrency actually bought --
a ratio close to `N_CONCURRENT` means the network-mounted ACAF reads scaled
fine; a ratio well below that means they were contending, and cuts against
pushing `N_CONCURRENT` higher without first testing on `dsub` (separate
machines, separate network paths per job) instead.

In [ ]:
import pandas as pd

summary = pd.DataFrame(results).sort_values("chr")
display(summary)

n_genome_wide_total = summary["n_thin"].sum()
naive_serial_hours = summary["elapsed_sec"].sum() / 3600
print(f"Genome-wide total (thinned): {n_genome_wide_total:,}")
print(f"Naive serial estimate: {naive_serial_hours:.2f} hours")
print(f"Actual wall-clock: {overall_elapsed/3600:.2f} hours")
print(f"Effective speedup: {naive_serial_hours / (overall_elapsed/3600):.2f}x (vs. N_CONCURRENT={N_CONCURRENT})")

## Merge chromosomes into a genome-wide pfile

GRM construction needs one unified panel across the whole autosomal genome --
relatedness for a pair of samples sums their genotype correlation across
every variant genome-wide, not per chromosome -- so the 22 QC'd+thinned
per-chromosome filesets (same `N` samples, disjoint variant sets) need to be
concatenated into a single pfile before `03_grm_shards`'s GRM step can use
them. Copies each chromosome's persisted trio from `BUCKET_DIR` back to local
scratch first (same convention as everywhere else in this pipeline --
sequential plink2 I/O against the gcsfuse-mounted bucket is much slower than
local disk), merges locally via `--pmerge-list`, then persists the merged
result back to the bucket.

Also exports a `--make-bed` (PLINK1 binary) copy: `GRM-pairs/grm_bin_sharded`'s
row-range recovery is calibrated to **PLINK 1.9's** own `--parallel` split
algorithm specifically (see that tool's README), and PLINK 1.9 doesn't read
pgen -- only `.bed`/`.bim`/`.fam`. Both formats get persisted, since the pgen
is the more useful canonical record and the bed is what the GRM step actually
needs.

In [ ]:
import shutil

for chr_num in range(1, 23):
    for ext in ("pgen", "pvar", "psam"):
        bucket_path = os.path.join(BUCKET_DIR, f"chr{chr_num}_round2b_thinned.{ext}")
        local_path = os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_round2b_thinned.{ext}")
        assert os.path.isfile(bucket_path), f"missing persisted chromosome output: {bucket_path!r} -- rerun the driver cell for chr{chr_num} first"
        if not os.path.isfile(local_path):
            shutil.copy(bucket_path, local_path)

print("All 22 chromosome trios present in local scratch.")

In [ ]:
MERGE_LIST_PATH = os.path.join(LOCAL_WORK_DIR, "chr_merge_list.txt")
with open(MERGE_LIST_PATH, "w") as f:
    for chr_num in range(1, 23):
        f.write(os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_round2b_thinned") + "\n")

MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, "genome_wide_round2b_thinned")
BED_PREFIX = f"{MERGED_PREFIX}_bed"

print(MERGE_LIST_PATH)
print(MERGED_PREFIX)

In [ ]:
%%bash -s "$MERGE_LIST_PATH" "$MERGED_PREFIX" "$BED_PREFIX"
set -e
MERGE_LIST_PATH=$1
MERGED_PREFIX=$2
BED_PREFIX=$3

time plink2 \
  --pmerge-list "$MERGE_LIST_PATH" \
  --make-pgen \
  --out "$MERGED_PREFIX"

echo "Genome-wide variant count:"
grep -vc '^##' "${MERGED_PREFIX}.pvar"
echo "Sample count:"
wc -l < "${MERGED_PREFIX}.psam"

# PLINK 1.9 (the GRM step) doesn't read pgen -- export a bed/bim/fam copy too
time plink2 \
  --pfile "$MERGED_PREFIX" \
  --make-bed \
  --out "$BED_PREFIX"

ls -lh "${MERGED_PREFIX}".* "${BED_PREFIX}".*

In [ ]:
%%bash -s "$MERGED_PREFIX" "$BED_PREFIX" "$BUCKET_DIR"
set -e
MERGED_PREFIX=$1
BED_PREFIX=$2
BUCKET_DIR=$3

mkdir -p "$BUCKET_DIR"
cp "${MERGED_PREFIX}".pgen "${MERGED_PREFIX}".pvar "${MERGED_PREFIX}".psam "${MERGED_PREFIX}".log "$BUCKET_DIR/"
cp "${BED_PREFIX}".bed "${BED_PREFIX}".bim "${BED_PREFIX}".fam "${BED_PREFIX}".log "$BUCKET_DIR/"

ls -lh "$BUCKET_DIR"/"$(basename "$MERGED_PREFIX")".* "$BUCKET_DIR"/"$(basename "$BED_PREFIX")".*